In [ ]:
!pip install mne --quiet

import os
import numpy as np
import pandas as pd
import mne
from ssqueezepy import ssq_cwt
from scipy.integrate import simpson

# EEG Bands (Hz)
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45)
}

def compute_band_energy(Tx, freqs):
    band_energy = {}
    power = np.abs(Tx)**2
    for band, (fmin, fmax) in BANDS.items():
        idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
        if len(idx) == 0:
            band_energy[band] = 0
        else:
            energy = np.trapz(power[idx, :], dx=1.0, axis=0)
            band_energy[band] = np.mean(energy)
    return band_energy

def extract_sct_features(file_path, label):
    raw = mne.io.read_raw_eeglab(file_path, preload=True)
    raw.pick("eeg")
    raw.set_montage("standard_1020")
    raw.resample(128)
    sfreq = raw.info['sfreq']
    data, _ = raw[:, :int(sfreq * 30)]  # use first 10 seconds

    features = {}
    for i, ch_name in enumerate(raw.ch_names):
        signal = data[i, :]
        Tx, freqs, *_ = ssq_cwt(signal, fs=sfreq, fmin=1, fmax=45)
        band_feats = compute_band_energy(Tx, freqs)
        for band, value in band_feats.items():
            features[f"{ch_name}_{band}"] = value
    features['label'] = label
    return features


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install mne ssqueezepy --quiet


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import pandas as pd
import mne
from ssqueezepy import ssq_cwt
import matplotlib.pyplot as plt

# Define EEG bands
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45)
}

def compute_band_energy(Tx, freqs):
    band_energy = {}
    power = np.abs(Tx)**2
    for band, (fmin, fmax) in BANDS.items():
        idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
        if len(idx) == 0:
            print(f"⚠️ No freqs for band {band}")
            band_energy[band] = 0
        else:
            energy = np.trapz(power[idx, :], dx=1.0, axis=0)
            band_energy[band] = np.mean(energy)
    return band_energy

def extract_sct_features(file_path, label):
    raw = mne.io.read_raw_eeglab(file_path, preload=True)
    raw.pick("eeg")
    raw.set_montage("standard_1020")
    raw.resample(128)
    sfreq = raw.info['sfreq']
    data, _ = raw[:, :int(sfreq * 10)]  # first 10 seconds only

    features = {}
    for i, ch_name in enumerate(raw.ch_names):
        signal = data[i, :]
        Tx, _, ssq_scales, *_ = ssq_cwt(signal, fs=sfreq)

        # Convert scales to freqs (Hz)
        freqs = sfreq / ssq_scales

        # Keep only valid 1–45 Hz range
        valid_idx = np.where((freqs >= 1) & (freqs <= 45))[0]
        if len(valid_idx) == 0:
            print(f"⚠️ {ch_name} has no valid freqs in 1–45 Hz (after scale-to-freq)")
            continue

        Tx = Tx[valid_idx, :]
        freqs = freqs[valid_idx]

        band_feats = compute_band_energy(Tx, freqs)
        for band, val in band_feats.items():
            features[f"{ch_name}_{band}"] = val
    features['label'] = label
    return features


In [ ]:
def process_folder(folder_path, label, output_csv_path):
    all_features = []
    for file in sorted(os.listdir(folder_path)):
        if file.endswith(".set"):
            full_path = os.path.join(folder_path, file)
            try:
                print(f"✅ Processing: {file}")
                feats = extract_sct_features(full_path, label)
                print(f"➡️  {file} features: {len(feats)} keys")
                all_features.append(feats)
            except Exception as e:
                print(f"❌ Failed on {file}: {e}")

    df = pd.DataFrame(all_features)

# Normalize only numeric (feature) columns
    features_only = df.drop(columns=['label'])
    df_normalized = (features_only - features_only.mean()) / features_only.std()

# Add label back after normalization
    df_normalized['label'] = df['label']

# Save to CSV
    df_normalized.to_csv(output_csv_path, index=False)
    print(f"💾 Saved normalized features to {output_csv_path}")

In [ ]:
import os

In [ ]:
ad_folder = "/content/drive/MyDrive/AD_001_035"
ad_csv_out = "/content/drive/MyDrive/features_AD/features_AD_02.csv"  # make sure this folder exists
os.makedirs(os.path.dirname(ad_csv_out), exist_ok=True)

process_folder(ad_folder, label="AD", output_csv_path=ad_csv_out)


✅ Processing: sub-001_task-eyesclosed_eeg.set


/tmp/ipython-input-6-4270813107.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  energy = np.trapz(power[idx, :], dx=1.0, axis=0)


➡️  sub-001_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-002_task-eyesclosed_eeg.set
➡️  sub-002_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-003_task-eyesclosed_eeg.set
➡️  sub-003_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-004_task-eyesclosed_eeg.set
➡️  sub-004_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-005_task-eyesclosed_eeg.set
➡️  sub-005_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-006_task-eyesclosed_eeg.set
➡️  sub-006_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-007_task-eyesclosed_eeg.set
➡️  sub-007_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-008_task-eyesclosed_eeg.set
➡️  sub-008_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-009_task-eyesclosed_eeg.set
➡️  sub-009_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-010_task-eyesclosed_eeg.set
➡️  sub-010_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-011_task-eyesclosed_eeg.set


In [ ]:
# === CN Patients ===
cn_folder = "/content/drive/MyDrive/CN_036_065"
cn_csv_out = "/content/drive/MyDrive/features_CN/features_CN.csv"
os.makedirs(os.path.dirname(cn_csv_out), exist_ok=True)
process_folder(cn_folder, label="CN", output_csv_path=cn_csv_out)

✅ Processing: sub-036_task-eyesclosed_eeg.set


/tmp/ipython-input-6-4270813107.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  energy = np.trapz(power[idx, :], dx=1.0, axis=0)


➡️  sub-036_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-037_task-eyesclosed_eeg.set
➡️  sub-037_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-038_task-eyesclosed_eeg.set
➡️  sub-038_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-039_task-eyesclosed_eeg.set
➡️  sub-039_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-040_task-eyesclosed_eeg.set
➡️  sub-040_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-041_task-eyesclosed_eeg.set
➡️  sub-041_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-042_task-eyesclosed_eeg.set
➡️  sub-042_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-043_task-eyesclosed_eeg.set
➡️  sub-043_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-044_task-eyesclosed_eeg.set
➡️  sub-044_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-045_task-eyesclosed_eeg.set
➡️  sub-045_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-046_task-eyesclosed_eeg.set


In [ ]:
# === FTD Patients ===
ftd_folder = "/content/drive/MyDrive/FTD_065_088"
ftd_csv_out = "/content/drive/MyDrive/features_FTD/features_FTD.csv"
os.makedirs(os.path.dirname(ftd_csv_out), exist_ok=True)
process_folder(ftd_folder, label="FTD", output_csv_path=ftd_csv_out)

✅ Processing: sub-066_task-eyesclosed_eeg.set


/tmp/ipython-input-6-4270813107.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  energy = np.trapz(power[idx, :], dx=1.0, axis=0)


➡️  sub-066_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-067_task-eyesclosed_eeg.set
➡️  sub-067_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-068_task-eyesclosed_eeg.set
➡️  sub-068_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-069_task-eyesclosed_eeg.set
➡️  sub-069_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-070_task-eyesclosed_eeg.set
➡️  sub-070_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-071_task-eyesclosed_eeg.set
➡️  sub-071_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-072_task-eyesclosed_eeg.set
➡️  sub-072_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-073_task-eyesclosed_eeg.set
➡️  sub-073_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-074_task-eyesclosed_eeg.set
➡️  sub-074_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-075_task-eyesclosed_eeg.set
➡️  sub-075_task-eyesclosed_eeg.set features: 96 keys
✅ Processing: sub-076_task-eyesclosed_eeg.set


In [ ]:
import pandas as pd

ad = pd.read_csv("/content/drive/MyDrive/features_AD/features_AD_02.csv")
cn = pd.read_csv("/content/drive/MyDrive/features_CN/features_CN.csv")
ftd = pd.read_csv("/content/drive/MyDrive/features_FTD/features_FTD.csv")

combined = pd.concat([ad, cn, ftd], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
combined.to_csv("/content/drive/MyDrive/features_combined.csv", index=False)

print("✅ Combined features saved to: features_combined.csv")


✅ Combined features saved to: features_combined.csv
